# El Cifrado Vigenère:

### Concepto:

1. Se trata de una técnica de cifrado por sustitución polialfabética.

2. Se basa en la existencia de una palabra clave que determina el desplazamiento de cada letra del texto original.

3. Cada letra del texto se cifra sumando el valor numérico de la letra correspondiente de la clave, repitiendo la clave según sea necesario.

### Ejemplo:
[Fuente](https://www.youtube.com/watch?v=8Pb5U64iwV4&list=WL&index=9&pp=gAQBiAQB)

![esquema explicativo](./assets/asset1.png)


El cifrado Vigenère es polialfabético. La fórmula matemática para cifrar el i-ésimo carácter del mensaje MSJ con la clave K de longitud m es:


- Mensaje: MSJ
- Letra i-ésima del mensaje: MSJi​
- Clave: K
- Letra i-ésima de la clave: Ki​
- Longitud de la clave: m
- Resultado Cifrado: Ci​
- mod 26: Operación módulo 26 (número de letras en el alfabeto Español sin la 'ñ')


### Ci​=(MSJi​+Ki(mod m)​)(mod 26)


In [ ]:
from laboratorio.Funciones_aux import normalizar_texto
from laboratorio.Kasiski import encontrar_clave_kasiski_icm
from laboratorio.Indice_Coincidencias import encontrar_clave_por_frecuencias

In [3]:
# ciframos con vigenere el texto a partir de la clave 
def cifrar_vigenere(msj, k):
    msj = normalizar_texto(msj)
    k = normalizar_texto(k)
    resultado = ""
    
    for i, msji in enumerate(msj):
        # Añadimos esta comprobacion para evitar que se introduccan caracteres no alfabeticos -> . , etc
        if msji.isalpha():
            # Obtenemos el valor de la posición i de la clave
            ki = k[i % len(k)] # Ki(mod(m)
            
            # Convertimos el char del texto y de la clave a su correspondiente valor ascii y restamos
            # el valor de la primera posición(A) para tener los valores entre 0 y 25
            msj_num = ord(msji) - ord('A')
            k_num = ord(ki) - ord('A')
            
            # Aplicamos el cifrado
            ci_num = (msj_num + k_num) % 26
            
            # Convertimos el número cifrado a char de nuevo y lo añadimos al resultado
            resultado += chr(ci_num + ord('A'))
    
    return resultado

# desciframos con vigenere el texto a partir de la clave
def descifrar_vigenere(c, k):
    c = normalizar_texto(c)
    k = normalizar_texto(k)
    resultado = ""
    
    for i, ci in enumerate(c):
        if ci.isalpha():
            # Obtenemos el valor de la posición i de la clave
            ki = k[i % len(k)]
            
            # Convertimos el char del texto y de la clave a su correspondiente valor ascii y restamos
            # el valor de la primera posición(A) para tener los valores entre 0 y 25            
            ci_num = ord(ci) - ord('A')
            ki_num = ord(ki) - ord('A')
            
            # Aplicamos descifrado
            msj_i_num = (ci_num - ki_num) % 26
            
            # Convertimos el número cifrado a char de nuevo y lo añadimos al resultado
            resultado += chr(msj_i_num + ord('A'))
        else:
            resultado += ci
    
    return resultado

Vamos a comprobar que esto funcione con el ejemplo:

In [4]:
mensaje = "yellow"
clave = "dog"

cifrado = cifrar_vigenere(mensaje, clave)
print(f"Mensaje original: {mensaje}")
print(f"Clave:            {clave}")
print(f"Cifrado:          {cifrado}") 

# Verificación inversa
descifrado = descifrar_vigenere(cifrado, clave)
print(f"Descifrado:       {descifrado}")

# El resultado esperado para 'yellow' con clave 'dog' es 'BSROCC'

Mensaje original: yellow
Clave:            dog
Cifrado:          BSROCC
Descifrado:       YELLOW


## Laboratorio de pruebas:

### Estimación de la longitud de la clave mediante el método Kasiski:
Búsqueda de patrones: Se rastrea el texto cifrado en busca de secuencias de letras idénticas (generalmente de 3 o más caracteres) que se repitan.

Fundamento: Si una secuencia se repite, es muy probable que corresponda al mismo fragmento de texto claro cifrado con la misma parte de la clave.

Cálculo de distancias: Se cuenta el número de caracteres que separan el inicio de una secuencia repetida y el inicio de su siguiente aparición.

Factorización: La longitud de la clave debe ser un divisor de esa distancia (o la distancia es un múltiplo de la longitud de la clave).

Análisis estadístico: Se calculan los divisores de todas las distancias halladas. El divisor que aparezca con mayor frecuencia es la longitud probable de la clave (m).

![esquema subsecuencias](./assets/asset2.png)

Ejemplo: Estimando longitud de clave con Kasiski

In [ ]:
import laboratorio.Kasiski

cifrado_largo = "UECWKDVLOTTVACKTPVGEZQMDAMRNPDDUXLBUICAMRHOECBHSPQLVIWOFFEAILPNTESMLDRUURIFAEQTTPXADWIAWLACCRPBHSRZIVQWOFROGTTNNXEVIVIBPDTTGAHVIACLAYKGJIEQHGECMESNNOCTHSGGNVWTQHKBPRHMVUOYWLIAFIRIGDBOEBQLIGWARQHNLOISQKEPEIDVXXNETPAXNZGDXWWEYQCTIGONNGJVHSQGEATHSYGSDVVOAQCXLHSPQMDMETRTMDUXTEQQJMFAEEAAIMEZREGIMUECICBXRVQRSMENNWTXTNSRNBPZHMRVRDYNECGSPMEAVTENXKEQKCTTHSPCMQQHSQGTXMFPBGLWQZRBOEIZHQHGRTOBSGTATTZRNFOSMLEDWESIWDRNAPBFOFHEGIXLFVOGUZLNUSRCRAZGZRTTAYFEHKHMCQNTZLENPUCKBAYCICUBNRPCXIWEYCSIMFPRUTPLXSYCBGCCUYCQJMWIEKGTUBRHVATTLEKVACBXQHGPDZEANNTJZTDRNSDTFEVPDXKTMVNAIQMUQNOHKKOAQMTBKOFSUTUXPRTMXBXNPCLRCEAEOIAWGGVVUSGIOEWLIQFOZKSPVMEBLOHLXDVCYSMGOPJEFCXMRUIGDXNCCRPMLCEWTPZMOQQSAWLPHPTDAWEYJOGQSOAVERCTNQQEAVTUGKLJAXMRTGTIEAFWPTZYIPKESMEAFCGJILSBPLDABNFVRJUXNGQSWIUIGWAAMLDRNNPDXGNPTTGLUHUOBMXSPQNDKBDBTEECLECGRDPTYBVRDATQHKQJMKEFROCLXNFKNSCWANNAHXTRGKCJTTRRUEMQZEAEIPAWEYPAJBBLHUEHMVUNFRPVMEDWEKMHRREOGZBDBROGCGANIUYIBNZQVXTGORUUCUTNBOEIZHEFWNBIGOZGTGWXNRHERBHPHGSIWXNPQMJVBCNEIDVVOAGLPONAPWYPXKEFKOCMQTRTIDZBNQKCPLTTNOBXMGLNRRDNNNQKDPLTLNSUTAXMNPTXMGEZKAEIKAGQ"

longitud_clave = laboratorio.Kasiski.estimar_longitud_clave_kasiski(cifrado_largo)
print(f"Longitud de clave estimada (Indice de Coincidencias): {longitud_clave}")

Longitud de clave estimada (Indice de Coincidencias): [7, 3, 2, 14, 5, 4, 9, 13, 15, 6, 17, 8, 10, 11, 16, 20, 19, 12, 18]


### Estimación de la longitud de la clave mediante el método del Índice de Coincidencia:

El Índice de Coincidencia (IC) mide la probabilidad de que dos letras seleccionadas al azar del texto cifrado sean iguales.

Procedimiento:

1. División del texto cifrado: Se divide el texto cifrado en varias subsecuencias, cada una correspondiente a una posición específica dentro de la clave. Por ejemplo, si la clave tiene longitud m, se crean m subsecuencias.

2. Cálculo del IC para cada subsecuencia: Se calcula el Índice de Coincidencia para cada una de las subsecuencias obtenidas. Esto implica contar la frecuencia de cada letra en la subsecuencia y aplicar la fórmula del IC.

3. Comparación con valores esperados: Se comparan los valores de IC obtenidos con los valores típicos para el idioma del texto original. Un IC cercano al esperado para el idioma sugiere que la longitud de la clave es correcta.

In [27]:
cifrado_largo = "UECWKDVLOTTVACKTPVGEZQMDAMRNPDDUXLBUICAMRHOECBHSPQLVIWOFFEAILPNTESMLDRUURIFAEQTTPXADWIAWLACCRPBHSRZIVQWOFROGTTNNXEVIVIBPDTTGAHVIACLAYKGJIEQHGECMESNNOCTHSGGNVWTQHKBPRHMVUOYWLIAFIRIGDBOEBQLIGWARQHNLOISQKEPEIDVXXNETPAXNZGDXWWEYQCTIGONNGJVHSQGEATHSYGSDVVOAQCXLHSPQMDMETRTMDUXTEQQJMFAEEAAIMEZREGIMUECICBXRVQRSMENNWTXTNSRNBPZHMRVRDYNECGSPMEAVTENXKEQKCTTHSPCMQQHSQGTXMFPBGLWQZRBOEIZHQHGRTOBSGTATTZRNFOSMLEDWESIWDRNAPBFOFHEGIXLFVOGUZLNUSRCRAZGZRTTAYFEHKHMCQNTZLENPUCKBAYCICUBNRPCXIWEYCSIMFPRUTPLXSYCBGCCUYCQJMWIEKGTUBRHVATTLEKVACBXQHGPDZEANNTJZTDRNSDTFEVPDXKTMVNAIQMUQNOHKKOAQMTBKOFSUTUXPRTMXBXNPCLRCEAEOIAWGGVVUSGIOEWLIQFOZKSPVMEBLOHLXDVCYSMGOPJEFCXMRUIGDXNCCRPMLCEWTPZMOQQSAWLPHPTDAWEYJOGQSOAVERCTNQQEAVTUGKLJAXMRTGTIEAFWPTZYIPKESMEAFCGJILSBPLDABNFVRJUXNGQSWIUIGWAAMLDRNNPDXGNPTTGLUHUOBMXSPQNDKBDBTEECLECGRDPTYBVRDATQHKQJMKEFROCLXNFKNSCWANNAHXTRGKCJTTRRUEMQZEAEIPAWEYPAJBBLHUEHMVUNFRPVMEDWEKMHRREOGZBDBROGCGANIUYIBNZQVXTGORUUCUTNBOEIZHEFWNBIGOZGTGWXNRHERBHPHGSIWXNPQMJVBCNEIDVVOAGLPONAPWYPXKEFKOCMQTRTIDZBNQKCPLTTNOBXMGLNRRDNNNQKDPLTLNSUTAXMNPTXMGEZKAEIKAGQ"
longitud_clave = laboratorio.Indice_Coincidencias.estimar_longitud_clave_ic(cifrado_largo)

clave = laboratorio.Indice_Coincidencias.encontrar_clave_por_frecuencias(cifrado_largo, longitud_clave)
print(f"Clave estimada: {clave}")

Clave estimada: CAPITAN


Analizando el ejemplo del 3

Se trata de una cadena cifrada con Vigenère:

Para obtener la longitud de la clave, usamos el metodo estimar_longitud_clave_ic del indice de coincidencias:

Para obtener la clave, a partir del dato estimación de longitud de la clave, usamos la fuinción encontrar_clave_por_frecuencias.

Para obtener el texto de vuelta aplicamos vigenere basico para descrifrar el texto con la clave obtenida.

In [28]:
cifrado_largo = "UECWKDVLOTTVACKTPVGEZQMDAMRNPDDUXLBUICAMRHOECBHSPQLVIWOFFEAILPNTESMLDRUURIFAEQTTPXADWIAWLACCRPBHSRZIVQWOFROGTTNNXEVIVIBPDTTGAHVIACLAYKGJIEQHGECMESNNOCTHSGGNVWTQHKBPRHMVUOYWLIAFIRIGDBOEBQLIGWARQHNLOISQKEPEIDVXXNETPAXNZGDXWWEYQCTIGONNGJVHSQGEATHSYGSDVVOAQCXLHSPQMDMETRTMDUXTEQQJMFAEEAAIMEZREGIMUECICBXRVQRSMENNWTXTNSRNBPZHMRVRDYNECGSPMEAVTENXKEQKCTTHSPCMQQHSQGTXMFPBGLWQZRBOEIZHQHGRTOBSGTATTZRNFOSMLEDWESIWDRNAPBFOFHEGIXLFVOGUZLNUSRCRAZGZRTTAYFEHKHMCQNTZLENPUCKBAYCICUBNRPCXIWEYCSIMFPRUTPLXSYCBGCCUYCQJMWIEKGTUBRHVATTLEKVACBXQHGPDZEANNTJZTDRNSDTFEVPDXKTMVNAIQMUQNOHKKOAQMTBKOFSUTUXPRTMXBXNPCLRCEAEOIAWGGVVUSGIOEWLIQFOZKSPVMEBLOHLXDVCYSMGOPJEFCXMRUIGDXNCCRPMLCEWTPZMOQQSAWLPHPTDAWEYJOGQSOAVERCTNQQEAVTUGKLJAXMRTGTIEAFWPTZYIPKESMEAFCGJILSBPLDABNFVRJUXNGQSWIUIGWAAMLDRNNPDXGNPTTGLUHUOBMXSPQNDKBDBTEECLECGRDPTYBVRDATQHKQJMKEFROCLXNFKNSCWANNAHXTRGKCJTTRRUEMQZEAEIPAWEYPAJBBLHUEHMVUNFRPVMEDWEKMHRREOGZBDBROGCGANIUYIBNZQVXTGORUUCUTNBOEIZHEFWNBIGOZGTGWXNRHERBHPHGSIWXNPQMJVBCNEIDVVOAGLPONAPWYPXKEFKOCMQTRTIDZBNQKCPLTTNOBXMGLNRRDNNNQKDPLTLNSUTAXMNPTXMGEZKAEIKAGQ"
longitud_clave = laboratorio.Indice_Coincidencias.estimar_longitud_clave_ic(cifrado_largo)
print(f"Longitud de clave estimada (IC): {longitud_clave}")

clave = laboratorio.Indice_Coincidencias.encontrar_clave_por_frecuencias(cifrado_largo, longitud_clave)
print(f"Clave estimada: {clave}")

print("Texto obtenido")
descifrado = descifrar_vigenere(cifrado_largo, clave)
print(descifrado)

Longitud de clave estimada (IC): 7
Clave estimada: CAPITAN
Texto obtenido
SENORDIJOELCAPITANNEMOMOSTRANDOMELOSINSTRUMENTOSCOLGADOSDELASPAREDESDESUCAMAROTEHEAQUILOSAPARATOSEXIGIDOSPORLANAVEGACIONDELNAUTILUSALIGUALQUEENELSALONLOSTENGOAQUIBAJOMISOJOSINDICANDOMEMISITUACIONYMIDIRECCIONEXACTASENMEDIODELOCEANOALGUNOSDEELLOSLESONCONOCIDOSCOMOELTERMOMETROQUEMARCALATEMPERATURAINTERIORDELNAUTILUSELBAROMETROQUEPESAELAIREYPREDICELOSCAMBIOSDETIEMPOELHIGROMETROQUEREGISTRAELGRADODESEQUEDADDELAATMOSFERAELSTORMGLASSCUYAMEZCLAALDESCOMPONERSEANUNCIALAINMINENCIADELASTEMPESTADESLABRUJULAQUEDIRIGEMIRUTAELSEXTANTEQUEPORLAALTURADELSOLMEINDICAMILATITUDLOSCRONOMETROSQUEMEPERMITENCALCULARMILONGITUDYPORULTIMOMISANTEOJOSDEDIAYDENOCHEQUEMESIRVENPARAESCRUTARTODOSLOSPUNTOSDELHORIZONTECUANDOELNAUTILUSEMERGEALASUPERFICIEDELASAGUASSONLOSINSTRUMENTOSHABITUALESDELNAVEGANTEYSUUSOMEESCONOCIDOREPUSEPEROHAYOTROSAQUIQUERESPONDENSINDUDAALASPARTICULARESEXIGENCIASDELNAUTILUSESECUADRANTEQUEVEORECORRIDOPORUNAAGUJAINMOVILNOESUNMANOME